# Environment Exploration

This notebook explores the Ms. Pac-Man environment, its observations, actions, and preprocessing pipeline.

## 1. Setup

In [ ]:
# Install dependencies (for Colab)
# !pip install gymnasium[atari] ale-py opencv-python numpy matplotlib

In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
import sys
sys.path.append('..')

from src.environment import make_atari_env, create_env

## 2. Raw Environment

In [ ]:
# Create raw environment
raw_env = gym.make("ALE/MsPacman-v5")

print(f"Observation space: {raw_env.observation_space}")
print(f"Action space: {raw_env.action_space}")
print(f"Number of actions: {raw_env.action_space.n}")
print(f"\nAction meanings: {raw_env.unwrapped.get_action_meanings()}")

In [ ]:
# Reset and visualize raw observation
obs, info = raw_env.reset(seed=42)

plt.figure(figsize=(8, 6))
plt.imshow(obs)
plt.title(f"Raw Observation: {obs.shape}")
plt.axis('off')
plt.show()

print(f"Observation shape: {obs.shape}")
print(f"Observation dtype: {obs.dtype}")
print(f"Value range: [{obs.min()}, {obs.max()}]")

## 3. Preprocessed Environment

In [ ]:
# Create preprocessed environment
env = make_atari_env("ALE/MsPacman-v5", seed=42)

print(f"Preprocessed observation space: {env.observation_space}")
print(f"Action space: {env.action_space}")

In [ ]:
# Reset and visualize preprocessed observation
obs, info = env.reset()

print(f"Observation shape: {obs.shape}")
print(f"Number of stacked frames: {obs.shape[0]}")
print(f"Frame size: {obs.shape[1]}x{obs.shape[2]}")

# Visualize all 4 stacked frames
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i in range(4):
    axes[i].imshow(obs[i], cmap='gray')
    axes[i].set_title(f"Frame {i+1}")
    axes[i].axis('off')
plt.suptitle("4 Stacked Frames (84x84 Grayscale)")
plt.tight_layout()
plt.show()

## 4. Preprocessing Pipeline Comparison

In [ ]:
# Show preprocessing steps
raw_obs, _ = raw_env.reset(seed=42)
proc_obs, _ = env.reset()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(raw_obs)
axes[0].set_title(f"Raw: {raw_obs.shape}\n(210x160 RGB)")
axes[0].axis('off')

axes[1].imshow(proc_obs[0], cmap='gray')
axes[1].set_title(f"Preprocessed: {proc_obs.shape}\n(4x84x84 Grayscale, Frame 1 shown)")
axes[1].axis('off')

plt.tight_layout()
plt.show()

print(f"Size reduction: {raw_obs.nbytes / proc_obs.nbytes:.1f}x smaller")

## 5. Action Space Exploration

In [ ]:
# Test all actions
action_meanings = raw_env.unwrapped.get_action_meanings()

print("Available actions:")
for i, action in enumerate(action_meanings):
    print(f"  {i}: {action}")

In [ ]:
# Take random actions and observe
obs, _ = env.reset()
total_reward = 0
steps = 0

for _ in range(100):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    steps += 1
    
    if terminated or truncated:
        break

print(f"Random policy performance:")
print(f"  Steps: {steps}")
print(f"  Total reward: {total_reward}")
print(f"  Average reward: {total_reward/steps:.3f}")

## 6. Reward Distribution

In [ ]:
# Collect rewards over multiple episodes
episode_rewards = []
episode_lengths = []

for episode in range(10):
    obs, _ = env.reset()
    episode_reward = 0
    episode_length = 0
    
    while True:
        action = env.action_space.sample()
        obs, reward, terminated, truncated, info = env.step(action)
        episode_reward += reward
        episode_length += 1
        
        if terminated or truncated:
            break
    
    episode_rewards.append(episode_reward)
    episode_lengths.append(episode_length)

print(f"\nRandom Policy Statistics (10 episodes):")
print(f"  Mean reward: {np.mean(episode_rewards):.2f} ± {np.std(episode_rewards):.2f}")
print(f"  Mean length: {np.mean(episode_lengths):.1f} ± {np.std(episode_lengths):.1f}")
print(f"  Reward range: [{min(episode_rewards):.0f}, {max(episode_rewards):.0f}]")

In [ ]:
# Plot reward distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].bar(range(len(episode_rewards)), episode_rewards)
axes[0].axhline(np.mean(episode_rewards), color='r', linestyle='--', label='Mean')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Total Reward')
axes[0].set_title('Episode Rewards (Random Policy)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].bar(range(len(episode_lengths)), episode_lengths)
axes[1].axhline(np.mean(episode_lengths), color='r', linestyle='--', label='Mean')
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('Episode Length')
axes[1].set_title('Episode Lengths (Random Policy)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Reward Clipping Effect

In [ ]:
# Compare with and without reward clipping
env_clipped = make_atari_env("ALE/MsPacman-v5", seed=42, clip_rewards=True)
env_unclipped = make_atari_env("ALE/MsPacman-v5", seed=42, clip_rewards=False)

print("Testing reward clipping...\n")

# Run same actions on both
np.random.seed(42)
obs_c, _ = env_clipped.reset(seed=42)
obs_u, _ = env_unclipped.reset(seed=42)

for step in range(100):
    action = np.random.randint(0, env.action_space.n)
    
    _, reward_c, done_c, _, _ = env_clipped.step(action)
    _, reward_u, done_u, _, _ = env_unclipped.step(action)
    
    if reward_u != 0:  # Only show non-zero rewards
        print(f"Step {step}: Unclipped={reward_u:.1f}, Clipped={reward_c:.1f}")
    
    if done_c or done_u:
        break

env_clipped.close()
env_unclipped.close()

## 8. Summary

Key takeaways:
- Raw observations are 210×160×3 RGB images
- Preprocessed observations are 4×84×84 grayscale (stacked frames)
- 9 discrete actions available
- Random policy achieves ~200 points on average
- Rewards are clipped to {-1, 0, 1} for stable learning
- Episodes typically last 200-500 steps with random policy

Next steps:
- Train DQN agent (notebook 02)
- Train PPO agent (notebook 03)
- Train A2C agent (notebook 04)
- Compare algorithms (notebook 05)

In [ ]:
# Cleanup
raw_env.close()
env.close()